In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import os

In [ ]:
# Download NLTK data files (run this once if not downloaded)
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Leibniz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Leibniz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Load data
property_df = pd.read_csv('QGIS_df.csv')  # property DataFrame
RR_CR_dct = {"condo": "RR", "house": "RR", "apartment":"RR", "commercial":"CR", "land":"CR"}
property_df["Property Type"] = property_df["Category"].map(RR_CR_dct)

cmci_data = pd.read_csv(os.path.join("Independent Data", "CMCI_2023_Shortlist.csv"))

# Set up stop words and punctuation for preprocessing
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

In [ ]:
print(property_df.columns)
print(cmci_data.columns)

Index(['Location', 'URL', 'SKU', 'Title', 'Price', 'Category', 'Num_Bedrooms',
       'Num_Bathrooms', 'Floor_Area', 'Land_Area', 'Description', 'Agent_Name',
       'Agent_Link', 'Agent_Verification', 'File', 'Latitude', 'Longitude',
       'LRTHubName', 'LRTHubDist', 'BusHubName', 'BusHubDist',
       'HospitalHubName', 'HospitalHubDist', 'MallHubName', 'MallHubDist',
       'SchoolHubName', 'SchoolHubDist', 'BusCount', 'HospitalCount',
       'MallCount', 'SchoolCount', 'Concat', 'Property Type'],
      dtype='object')
Index(['PROVINCE / LGU', 'employment_generation', 'financial_deepening',
       'local_economy_growth', 'local_economy_size',
       'presence_of_business_and_professional_organizations',
       'safety_compliant_business'],
      dtype='object')


In [ ]:
!pip install fuzzywuzzy

In [ ]:
from fuzzywuzzy import process

unique_property_locations = property_df["Location"].dropna().unique()
cmci_cities = cmci_data["PROVINCE / LGU"].dropna().unique().tolist()

# Function to find best match using fuzzy matching
def fuzzy_match(location, choices, threshold=80):
    result = process.extractOne(location, choices)  # Get best match
    if result:  # Check if a match was found
        match, score = result
        return match if score >= threshold else None
    return None  # Return None if no match found

# Perform fuzzy matching only on unique locations
matched_dict = {loc: fuzzy_match(loc, cmci_cities) for loc in unique_property_locations}
matched_dict.update({"Quezon City": 'Quezon (MM)', 'H-2, Dasmariñas': 'Dasmarinas'}) #Manually updated

# Map the matched results back to the original DataFrame
property_df["Matched_City"] = property_df["Location"].map(matched_dict)

# Show results
print(property_df[['Location', 'Matched_City']])

                        Location Matched_City
0               Amparo, Caloocan     Caloocan
1                       Caloocan     Caloocan
2               Amparo, Caloocan     Caloocan
3      Grace Park East, Caloocan     Caloocan
4      Grace Park East, Caloocan     Caloocan
...                          ...          ...
72014          Ugong, Valenzuela   Valenzuela
72015        Canumay, Valenzuela   Valenzuela
72016       Malanday, Valenzuela   Valenzuela
72017  Viente Reales, Valenzuela   Valenzuela
72018  Viente Reales, Valenzuela   Valenzuela

[72019 rows x 2 columns]


In [ ]:
# Check if there are any missing matches
property_df["Location"][property_df["Matched_City"].isna()]

Series([], Name: Location, dtype: object)

In [ ]:
merged_df = property_df.merge(cmci_data, left_on="Matched_City", right_on="PROVINCE / LGU", how="left")
merged_df.head()

                    Location  \
0           Amparo, Caloocan   
1                   Caloocan   
2           Amparo, Caloocan   
3  Grace Park East, Caloocan   
4  Grace Park East, Caloocan   

                                                 URL                SKU  \
0  https://www.lamudi.com.ph/projects/amparo-resi...  HO5CEE34F0060ECPH   
1  https://www.lamudi.com.ph/buy/metro-manila/cal...  CD66D1B50AECE7EPH   
2  https://www.lamudi.com.ph/buy/metro-manila/cal...  LA66C596CFA8463PH   
3  https://www.lamudi.com.ph/buy/metro-manila/cal...  CD66C1D00AA6DD1PH   
4  https://www.lamudi.com.ph/buy/metro-manila/cal...  CD66C6B155BF32BPH   

                                               Title       Price Category  \
0  7.2M single attached house and lot for sale at...   7200000.0    house   
1       Caloocan City RFO Condo beside Lrt Monumento   3100000.0    condo   
2  850sqm Vacant Lot in Makabud Street, Amparo, N...  17000000.0     land   
3  Pre Selling 2BR 97sqm in Caloocan City near A